# Random forests
Decision trees are good estimators for classification tasks, also performs enough well in regression tasks. However, can't learn complex patterns as they are weak learners, also overfitting is a common issue in deep trees. Once, somebody told "two heads thinks better than one" and he was right, by coincidence that's the premise of random forests. Random forests is a bagging ML algorithm that consists on training some decision trees to create a strong model capable of learning complex relationships.

## Core concepts
- **Ensemble:** Approach to create a strong model from a set of some weak learners combining their outputs to improve robustness.
- **Bagging:** An ensemble strategy that consists on training some individual estimators on random subsets of the original training set and the aggregate their individual predictions to form the model outcome.
- **Bootstraping:** A sampling technique that uses simple random sampling with replacement within instances into a dataset.
- **Decision tree:** A weak ML learner that works evaluating hierarchicaly ordered conditions into the sample following a decision path until reaching a leaf node to produce a model outcome.

## How it works?
1. Build $n$ individual decision trees.
3. From the training dataset create $n$ sample subsets using bootstrapping and pick random features to each one.
4. Assign a subset to every decision tree.
5. Train each decision tree independently.

### Classification
The model uses a majority voting strategy to select the class label. The sample is provided to each decision tree, after every model generates its prediction (selects a label for the instance), choose the most voted (the one with highest frequency) as model outcome.

In [1]:
from pandas import read_csv
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = RandomForestClassifier(n_estimators=20, criterion="entropy", max_depth=10, random_state=123)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

Accuracy: 84.750000 %
Weighted F1-Score: 84.519687 %


### Regression
Similarly to a single decision tree, it just averages the output of each decision tree.

In [2]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = RandomForestRegressor(n_estimators=4, max_depth=10, criterion="squared_error", random_state=123)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")

Mean squared error: 767588830.286500
Mean absolute error: 7339.217515


## Most relevant hyperparameters
- **`n_estimators`:** Number of trees in the forest.
- **`criterion`:** Function used to measure the quality of a split (`gini`, `entropy`, `mae`, `mse`, etc).
- **`max_depth`:** Maximum depth of the tree, if not specified, nodes are expanded until leaves are pure or all leaves contains less than `min_samples_split` samples.
- **`min_samples_split`:** Minimum number of samples required to split an internal node.
- **`min_samples_leaf`:** Minimun number of samples required to be a leaf node.
- **`max_features`:** The number of features to consider when looking the best split.
- **`boostrap`:** A flag for disabling bootstrapping (every tree will be trained with the whole dataser).

Note that `splitter` hyperparameter dissapeared because we got enough randomness sources. So strictly chooses the subsample that maximizes/minimizes the split criterion.

## Advantages
- All benefits inherited from decision tress.
- Estimators can be trained in parallel.
- Decrease the high variance that single decision trees have.
- Lower overfitting risk than decision trees.
- A better way for estimating feature importance.
## Disadvantages
- Some bias is introduced as variance reduction.
- Large forests are slow at training.
- Harder to explain than single decision trees.